# 🚦 Traffic Demand Prediction — Gridathon v2
**Flipkart × Bengaluru Traffic Police | HackerEarth**

## 🔍 Key Fix from v1 → v2
| | v1 (87.6 online) | v2 (improved) |
|---|---|---|
| Training data | Day49 only (7.8K, 9 timestamps) | Day48 + Day49 (77K, 96 timestamps) |
| Validation | Same-day KFold (optimistic) | Cross-day: train Day48 → val Day49 |
| Root cause | Zero overlap between train & test timestamps | Fixed |

**Metric:** `score = max(0, 100 * r2_score(actual, predicted))`

## 1. Imports & Setup

In [ ]:
!pip install lightgbm -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

## 2. Load Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

day48 = train[train['day']==48].copy()
day49 = train[train['day']==49].copy()

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"Day48 rows  : {len(day48)}")
print(f"Day49 rows  : {len(day49)}")
train.head()

## 3. Root Cause Analysis — Why v1 Scored Only 87.6

In [ ]:
# CRITICAL DISCOVERY: timestamp mismatch between train split and test
day49_ts = set(day49['timestamp'].unique())
test_ts  = set(test['timestamp'].unique())
day48_ts = set(day48['timestamp'].unique())

print("=== TIMESTAMP DISTRIBUTION ===")
print(f"Day49 train timestamps ({len(day49_ts)}): {sorted(day49_ts)}")
print()
print(f"Test timestamps ({len(test_ts)}): {sorted(test_ts)[:10]} ...")
print()
print(f"Overlap day49_train ∩ test: {len(day49_ts & test_ts)} ← ZERO OVERLAP!")
print(f"Overlap day48 ∩ test      : {len(day48_ts & test_ts)} ← ALL 47 TEST SLOTS")
print()
print("CONCLUSION: v1 trained on night timestamps (0:00–2:00) but test is")
print("            morning/day timestamps (2:15–13:45). Fix: train on Day48.")

In [ ]:
# Visualize the mismatch
fig, ax = plt.subplots(figsize=(14, 3))

def ts_to_min(ts):
    h, m = map(int, ts.split(':'))
    return h*60 + m

all_ts = sorted(day48['timestamp'].unique(), key=ts_to_min)
x = [ts_to_min(t) for t in all_ts]

d48_set  = set(day48['timestamp'].unique())
d49_set  = set(day49['timestamp'].unique())
test_set = set(test['timestamp'].unique())

colors = []
for t in all_ts:
    if t in test_set and t in d48_set:
        colors.append('green')
    elif t in d49_set:
        colors.append('red')
    elif t in d48_set:
        colors.append('steelblue')
    else:
        colors.append('gray')

ax.bar(x, [1]*len(x), color=colors, width=12)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='steelblue', label='Day48 only'),
    Patch(color='red',       label='Day49 train (v1 trained here)'),
    Patch(color='green',     label='Test timestamps (in Day48 ✅)'),
], loc='upper right')
ax.set_xlabel('Time of day (minutes from midnight)')
ax.set_title('Timestamp Coverage — v1 trained on red (0–120min) but test is green (135–825min)')
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
def parse_ts(df):
    df = df.copy()
    sp = df['timestamp'].str.split(':', expand=True)
    df['hour']         = sp[0].astype(int)
    df['minute']       = sp[1].astype(int)
    df['time_minutes'] = df['hour']*60 + df['minute']

    # Cyclical encoding (avoids midnight discontinuity)
    df['hour_sin'] = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos'] = np.cos(2*np.pi*df['hour']/24)
    df['time_sin'] = np.sin(2*np.pi*df['time_minutes']/1440)
    df['time_cos'] = np.cos(2*np.pi*df['time_minutes']/1440)

    # Rush-hour flags
    df['is_morning_rush'] = ((df['hour']>=7)  & (df['hour']<=9)).astype(int)
    df['is_evening_rush'] = ((df['hour']>=17) & (df['hour']<=19)).astype(int)
    df['is_night']        = ((df['hour']>=22) | (df['hour']<=5)).astype(int)
    df['is_day']          = ((df['hour']>=9)  & (df['hour']<=17)).astype(int)

    # Geohash spatial hierarchy
    df['geo_prefix3'] = df['geohash'].str[:3]
    df['geo_prefix4'] = df['geohash'].str[:4]
    df['geo_prefix5'] = df['geohash'].str[:5]

    # Categorical encoding
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather']  = df['Weather'].fillna('Unknown')
    df['LargeVehicles_bin'] = (df['LargeVehicles']=='Allowed').astype(int)
    df['Landmarks_bin']     = (df['Landmarks']=='Yes').astype(int)
    df['RoadType_enc'] = df['RoadType'].map({'Residential':0,'Street':1,'Highway':2,'Unknown':-1}).fillna(-1)
    df['Weather_enc']  = df['Weather'].map({'Sunny':0,'Rainy':1,'Foggy':2,'Snowy':3,'Unknown':-1}).fillna(-1)
    df['Temperature']  = df['Temperature'].fillna(df['Temperature'].median())
    return df

day48 = parse_ts(day48)
day49 = parse_ts(day49)
test  = parse_ts(test)
print("✅ Temporal features added")

## 5. Day48 Aggregation Statistics (Reference Features)

In [ ]:
def compute_stats(ref):
    """Compute all aggregation stats from a reference dataframe (Day48)."""
    geo    = ref.groupby('geohash')['demand'].agg(['mean','std','median','max','min','skew']).reset_index()
    geo.columns = ['geohash','geo_mean','geo_std','geo_median','geo_max','geo_min','geo_skew']

    ts     = ref.groupby('timestamp')['demand'].agg(['mean','std','median']).reset_index()
    ts.columns = ['timestamp','ts_mean','ts_std','ts_median']

    geo_ts = ref.groupby(['geohash','timestamp'])['demand'].mean().reset_index()
    geo_ts.columns = ['geohash','timestamp','geo_ts_mean']

    p3  = ref.groupby('geo_prefix3')['demand'].mean().reset_index(); p3.columns  = ['geo_prefix3','p3_mean']
    p4  = ref.groupby('geo_prefix4')['demand'].mean().reset_index(); p4.columns  = ['geo_prefix4','p4_mean']
    p5  = ref.groupby('geo_prefix5')['demand'].mean().reset_index(); p5.columns  = ['geo_prefix5','p5_mean']

    rts = ref.groupby(['RoadType','timestamp'])['demand'].mean().reset_index()
    rts.columns = ['RoadType','timestamp','road_ts_mean']

    wts = ref.groupby(['Weather','timestamp'])['demand'].mean().reset_index()
    wts.columns = ['Weather','timestamp','weather_ts_mean']

    lts = ref.groupby(['NumberofLanes','timestamp'])['demand'].mean().reset_index()
    lts.columns = ['NumberofLanes','timestamp','lanes_ts_mean']

    gh  = ref.groupby(['geohash','hour'])['demand'].mean().reset_index()
    gh.columns = ['geohash','hour','geo_hour_mean']

    p4ts = ref.groupby(['geo_prefix4','timestamp'])['demand'].mean().reset_index()
    p4ts.columns = ['geo_prefix4','timestamp','p4_ts_mean']

    p3ts = ref.groupby(['geo_prefix3','timestamp'])['demand'].mean().reset_index()
    p3ts.columns = ['geo_prefix3','timestamp','p3_ts_mean']

    return geo, ts, geo_ts, p3, p4, p5, rts, wts, lts, gh, p4ts, p3ts

stats48 = compute_stats(day48)
print("✅ Day48 stats computed")
print(f"  geo stats:    {stats48[0].shape}")
print(f"  ts stats:     {stats48[1].shape}")
print(f"  geo×ts stats: {stats48[2].shape}")

## 6. Apply Stats & Lag Feature

In [ ]:
def apply_stats(df, stats, day48_ref):
    """Merge all stats + lag feature from day48_ref into df."""
    geo,ts,geo_ts,p3,p4,p5,rts,wts,lts,gh,p4ts,p3ts = stats

    df = df.merge(geo,   on='geohash',                   how='left')
    df = df.merge(ts,    on='timestamp',                  how='left')
    df = df.merge(geo_ts,on=['geohash','timestamp'],      how='left')
    df = df.merge(p3,    on='geo_prefix3',                how='left')
    df = df.merge(p4,    on='geo_prefix4',                how='left')
    df = df.merge(p5,    on='geo_prefix5',                how='left')
    df = df.merge(rts,   on=['RoadType','timestamp'],     how='left')
    df = df.merge(wts,   on=['Weather','timestamp'],      how='left')
    df = df.merge(lts,   on=['NumberofLanes','timestamp'],how='left')
    df = df.merge(gh,    on=['geohash','hour'],           how='left')
    df = df.merge(p4ts,  on=['geo_prefix4','timestamp'],  how='left')
    df = df.merge(p3ts,  on=['geo_prefix3','timestamp'],  how='left')

    # Lag feature: day48 demand at same (geohash, timestamp)
    lag = day48_ref[['geohash','timestamp','demand']].copy()
    lag.columns = ['geohash','timestamp','demand_lag']
    df = df.merge(lag, on=['geohash','timestamp'], how='left')

    # Fill missing lag with best available proxy
    df['demand_lag'] = (df['demand_lag']
                        .fillna(df['geo_ts_mean'])
                        .fillna(df['p4_ts_mean'])
                        .fillna(df['ts_mean']))

    # Ratio & residual features
    df['lag_vs_geo']    = df['demand_lag'] / (df['geo_mean']    + 1e-9)
    df['lag_vs_ts']     = df['demand_lag'] / (df['ts_mean']     + 1e-9)
    df['lag_residual']  = df['demand_lag'] - df['geo_ts_mean'].fillna(df['ts_mean'])
    return df

day49_f = apply_stats(day49, stats48, day48)
test_f  = apply_stats(test,  stats48, day48)

# Day48 trains on itself — use geo_ts_mean as lag proxy
day48_f = apply_stats(day48, stats48, day48)
day48_f['demand_lag']  = day48_f['geo_ts_mean']
day48_f['lag_vs_geo']  = day48_f['geo_ts_mean'] / (day48_f['geo_mean'] + 1e-9)
day48_f['lag_vs_ts']   = day48_f['geo_ts_mean'] / (day48_f['ts_mean']  + 1e-9)
day48_f['lag_residual']= 0.0

lag_coverage = test_f['demand_lag'].notna().mean() * 100
print(f"✅ Features applied")
print(f"   Test lag coverage: {lag_coverage:.1f}%")
print(f"   Day48_f shape: {day48_f.shape}")
print(f"   Day49_f shape: {day49_f.shape}")
print(f"   Test_f shape : {test_f.shape}")

## 7. Feature List

In [ ]:
FEATURES = [
    # Temporal
    'hour', 'minute', 'time_minutes',
    'hour_sin', 'hour_cos', 'time_sin', 'time_cos',
    'is_morning_rush', 'is_evening_rush', 'is_night', 'is_day',
    # Road context
    'NumberofLanes', 'LargeVehicles_bin', 'Landmarks_bin',
    'RoadType_enc', 'Weather_enc', 'Temperature',
    # Geohash spatial stats (from Day48)
    'geo_mean', 'geo_std', 'geo_median', 'geo_max', 'geo_min', 'geo_skew',
    # Timestamp global stats (from Day48)
    'ts_mean', 'ts_std', 'ts_median',
    # Interaction stats
    'geo_ts_mean', 'geo_hour_mean',
    'p3_mean', 'p4_mean', 'p5_mean',
    'p3_ts_mean', 'p4_ts_mean',
    'road_ts_mean', 'weather_ts_mean', 'lanes_ts_mean',
    # Lag & ratios
    'demand_lag', 'lag_vs_geo', 'lag_vs_ts', 'lag_residual',
]

X_d48  = day48_f[FEATURES].fillna(0);  y_d48 = day48_f['demand']
X_d49  = day49_f[FEATURES].fillna(0);  y_d49 = day49_f['demand']
X_test = test_f[FEATURES].fillna(0)

# Combined for final training
X_all = pd.concat([X_d48, X_d49], axis=0).reset_index(drop=True)
y_all = pd.concat([y_d48, y_d49], axis=0).reset_index(drop=True)

print(f"Features     : {len(FEATURES)}")
print(f"X_d48 shape  : {X_d48.shape}")
print(f"X_d49 shape  : {X_d49.shape}")
print(f"X_all shape  : {X_all.shape}")
print(f"X_test shape : {X_test.shape}")
print(f"Nulls in X_all  : {X_all.isnull().sum().sum()}")
print(f"Nulls in X_test : {X_test.isnull().sum().sum()}")

## 8. Cross-Day Validation (Train Day48 → Validate Day49)

In [ ]:
# This is the HONEST validation: different day = real generalization
PARAMS = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'n_estimators'     : 5000,
    'learning_rate'    : 0.015,
    'num_leaves'       : 63,
    'min_child_samples': 30,
    'feature_fraction' : 0.7,
    'bagging_fraction' : 0.7,
    'bagging_freq'     : 5,
    'reg_alpha'        : 0.1,
    'reg_lambda'       : 0.3,
    'verbose'          : -1,
    'random_state'     : 42,
}

print("Training on Day48, validating on Day49 (cross-day)...")
m_cv = lgb.LGBMRegressor(**PARAMS)
m_cv.fit(
    X_d48, y_d48,
    eval_set=[(X_d49, y_d49)],
    callbacks=[
        lgb.early_stopping(300, verbose=False),
        lgb.log_evaluation(200)
    ]
)

val_pred = m_cv.predict(X_d49)
val_score = max(0, 100 * r2_score(y_d49, val_pred))
print(f"\n✅ Cross-day validation score : {val_score:.4f}")
print(f"   Best iteration             : {m_cv.best_iteration_}")

## 9. Validation Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(y_d49, val_pred, alpha=0.4, s=6, color='steelblue')
axes[0].plot([0,1],[0,1],'r--', lw=1.5)
axes[0].set_xlabel('Actual Demand (Day49)')
axes[0].set_ylabel('Predicted Demand')
axes[0].set_title(f'Cross-Day: Actual vs Predicted (Score={val_score:.2f})')

residuals = y_d49.values - val_pred
axes[1].hist(residuals, bins=80, color='coral', edgecolor='white')
axes[1].axvline(0, color='red', lw=1.5, ls='--')
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution (Day49 Validation)')

plt.tight_layout()
plt.show()

## 10. Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': m_cv.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(data=fi.head(20), x='importance', y='feature', palette='viridis')
plt.title('Top 20 Feature Importances (LightGBM)')
plt.tight_layout()
plt.show()

print(fi.head(15).to_string(index=False))

## 11. Final Model — Train on All Data

In [ ]:
# Train on Day48 + Day49 with best_iteration + buffer
final_iters = m_cv.best_iteration_ + 200
print(f"Training final model with {final_iters} iterations on all {len(X_all)} rows...")

m_final = lgb.LGBMRegressor(**{**PARAMS, 'n_estimators': final_iters})
m_final.fit(X_all, y_all, callbacks=[lgb.log_evaluation(period=-1)])

test_preds = np.clip(m_final.predict(X_test), 0, 1)
print("✅ Final model trained")
print(f"   Prediction range: [{test_preds.min():.4f}, {test_preds.max():.4f}]")
print(f"   Prediction mean : {test_preds.mean():.4f}")

## 12. Generate Submission

In [ ]:
submission = pd.DataFrame({
    'Index' : test['Index'],
    'demand': test_preds
})

submission.to_csv('submission_v2.csv', index=False)

print(f"Submission shape   : {submission.shape}")
print(f"Expected           : (41778, 2)")
print()
print(submission['demand'].describe())
print()
print(submission.head(10))

## 13. Summary

### What Was Wrong in v1
| Issue | Detail |
|---|---|
| Training data | Day49 train (7.8K rows, timestamps 0:00–2:00 only) |
| Test data | 41,778 rows, timestamps 2:15–13:45 |
| **Zero overlap** | Model trained on night, predicted morning/day |
| OOF inflated | 94.8 OOF vs 87.6 online — classic overfit signal |

### What v2 Does
| Component | Detail |
|---|---|
| **Training** | Day48 (69K rows, all 96 timestamps) + Day49 |
| **Validation** | Cross-day: train Day48 → val Day49 (honest) |
| **Lag feature** | Day48 demand at exact (geohash, timestamp) — 88.9% coverage |
| **New features** | `geo_hour_mean`, `p3_ts_mean`, `p4_ts_mean`, `lag_residual` |
| **Regularization** | Stronger (`num_leaves=63`, `min_child_samples=30`) |
| **Model** | LightGBM with early stopping on cross-day val |

### Further Improvements
- Ensemble with XGBoost / CatBoost
- Geohash lat/lng decoding for neighbor smoothing
- Optuna hyperparameter search